# YOLOv8n Chessboard Detection - Training & Evaluation

Train a YOLOv8n model to detect chessboards in GothamChess video frames.
Single class: `chessboard`.

In [ ]:
from pathlib import Path
import yaml

DATASET_YAML = Path("data/yolo_dataset/dataset.yaml")

with open(DATASET_YAML) as f:
    ds_config = yaml.safe_load(f)
print("Dataset config:", ds_config)

# Verify counts
for split in ["train", "val", "test"]:
    img_dir = Path(ds_config["path"]) / ds_config[split]
    n = len(list(img_dir.glob("*.png")))
    print(f"  {split}: {n} images")

## Train YOLOv8n

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATASET_YAML.resolve()),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
    degrees=5.0,
    translate=0.1,
    scale=0.3,
    flipud=0.0,
    fliplr=0.3,
    mosaic=0.5,
    mixup=0.1,
    # Output
    project="runs",
    name="yolov8n_chessboard",
    device=0,
)

## Evaluate on Test Set

In [ ]:
best_model = YOLO("runs/yolov8n_chessboard/weights/best.pt")

metrics = best_model.val(
    data=str(DATASET_YAML.resolve()),
    split="test",
)

print(f"mAP@0.5:     {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

## Visual Inspection

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random

test_images = sorted((Path(ds_config["path"]) / ds_config["test"]).glob("*.png"))
samples = random.sample(test_images, min(12, len(test_images)))

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
for ax, img_path in zip(axes.flat, samples):
    results = best_model.predict(str(img_path), verbose=False)
    annotated = results[0].plot()
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.stem[:30], fontsize=8)
    ax.axis("off")

for ax in axes.flat[len(samples):]:
    ax.axis("off")

plt.suptitle("YOLOv8n Chessboard Detection - Test Predictions", fontsize=14)
plt.tight_layout()
plt.savefig("runs/yolov8n_chessboard/test_predictions.png", dpi=150)
plt.show()

## Export to ONNX

In [ ]:
best_model.export(format="onnx", imgsz=640, simplify=True)
print("Exported to ONNX")

## Compare: YOLO vs Template Matching

Run both detectors on ChessAtlasBackend test images to compare.

In [ ]:
import sys
sys.path.insert(0, str(Path(".").resolve()))
from pathlib import Path

backend_test_dir = Path.home() / "Documents/GitHub/ChessAtlasBackend/test_images"
backend_frames = sorted(backend_test_dir.glob("*.png"))[:10]

print(f"Comparing on {len(backend_frames)} backend test images\n")

for img_path in backend_frames:
    results = best_model.predict(str(img_path), verbose=False)
    boxes = results[0].boxes
    if len(boxes) > 0:
        conf = boxes.conf[0].item()
        print(f"  {img_path.name}: YOLO detected (conf={conf:.3f})")
    else:
        print(f"  {img_path.name}: YOLO no detection")